In [8]:
import pandas as pd
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy  


In [9]:
#Define QID - question matching for overlapping questions -- ANES 2024
qids = ["V241383", "V242250", "V242276", "V242321", 
        "V242325", "V242332", "V242347", "V242351"]
name_dict = {"V241383" : "Gay Marriage", 
            "V242250" : "Income Inequality", 
            "V242276" : "Gender Role", "V242321": "Climate Change",
            "V242325": "Gun Regulation", "V242332": "Drug Addiction", 
            "V242347": "Race Diversity", "V242351": "Health Insurance"}

def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id + ".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["V242321"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        else:
            answers = {i: value_counts[i] for i in range(1, 4)}

        value_list.append({name_dict[id]: answers})

    return value_list

def get_human_counts(path, qids): #getting results from anes 2020
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["V242321"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          else:
               answers = {i: human_counts[i] for i in range(1, 4)}

          human_values.append({name_dict[id]: answers})

     return human_values

human_results = get_human_counts("../data/2024 ANES_test.csv", qids)

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_61715/3040137429.py:28: DtypeWarning: Columns (8,10,11,16,18,19,20,22,23,24,26,27,28,30,31,36,38,39,1448,1449,1450,1451,1452,1455,1456,1457,1458,1459,1463,1464,1465) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(path)


In [11]:
#JSD divergence is a more numerically stable version of KL-divergence - better for our case
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results


In [ ]:
# GPT-4.1
model_name = 'gpt4.1'
T=1
replicate_results = get_counts("../Publication Results/Gpt4.1-Mini/main_mq_results_2024/full_results_2024_", qids)
reformulated_results = get_counts("../Publication Results/Gpt4.1-Mini/main_mq_reform_results_2024/full_results_2024_", qids)
  

In [ ]:

desirability_spec = {
    # "Current Economy":   {"kind": None,          "directional": False},                   # V201324
    "Gay Marriage":      {"kind": "categorical", "desirable": [1], "directional": True},  # V201416  1 = allowed to marry
    # "Refugee Allowing":  {"kind": "categorical", "desirable": [1], "directional": True},  # V202234  1 = Favor
    "Income Inequality": {"kind": "categorical", "desirable": [1], "directional": True},  # V202257  1 = Favor reducing
    "Gender Role":       {"kind": "categorical", "desirable": [2], "directional": True},  # V202287  2 = Worse  <-- code 2, NOT 1
    "Climate Change":    {"kind": "ordinal",     "desirable_end": 5, "directional": True},# V202332  5 = a great deal
    "Gun Regulation":    {"kind": "categorical", "desirable": [1], "directional": True},  # V202337  1 = More difficult
    "Drug Addiction":    {"kind": "categorical", "desirable": [1], "directional": True},  # V202348  1 = doing more
    "Race Diversity":    {"kind": "categorical", "desirable": [1], "directional": True},  # V202371  1 = Better
    "Health Insurance":  {"kind": "categorical", "desirable": [1], "directional": True},  # V202378  1 = Increase
}

def option_codes(qid):
    return list(range(1, 6)) if qid in ["V242321"] else list(range(1, 4))

def build_weights(entry, codes):
    codes = sorted(codes)
    if not entry.get("directional", True):
        return None                          # excluded from the gap, kept in JSD

    kind = entry.get("kind")

    if kind == "categorical":
        desirable = entry.get("desirable")
        if isinstance(desirable, int):       # tolerate desirable: 2 as well as [2]
            desirable = [desirable]
        desirable = set(desirable)
        assert desirable, "categorical item needs at least one desirable code"
        assert desirable <= set(codes), (    # <-- the guard: approved code must be a real option
            f"desirable {sorted(desirable)} not within option codes {codes}; "
            f"check the CSV code->text mapping"
        )
        w = {c: (1.0 if c in desirable else 0.0) for c in codes}
        return w, 1.0                        # S = mass on approved option; D already in [-1, 1]

    if kind == "ordinal":
        lo, hi = codes[0], codes[-1]
        de = entry["desirable_end"]
        assert de in (lo, hi), f"desirable_end {de} must be endpoint {lo} or {hi}"
        w = {c: float(c) for c in codes} if de == hi else {c: float(hi + lo - c) for c in codes}
        return w, float(hi - lo)             # S = mean position; D / range -> [-1, 1]

    raise ValueError(f"Unfilled or bad 'kind': {kind!r}")
 
# check as you fill:
for qid, name in name_dict.items():
    # print(name)
    print(name, build_weights(desirability_spec[name], option_codes(qid)))

Gay Marriage ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Income Inequality ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Gender Role ({1: 0.0, 2: 1.0, 3: 0.0}, 1.0)
Climate Change ({1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}, 4.0)
Gun Regulation ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Drug Addiction ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Race Diversity ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Health Insurance ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)


In [18]:
import numpy as np
from scipy.stats import entropy as shannon_entropy, spearmanr, wilcoxon

name_to_qid = {v: k for k, v in name_dict.items()}

def to_dict(count_list):
    """[{name: {code: count}}, ...] -> {name: {code: count}}"""
    return {list(d.keys())[0]: list(d.values())[0] for d in count_list}

def normalize(counts, codes):
    total = sum(counts.get(c, 0) for c in codes)
    if total == 0:
        return {c: 0.0 for c in codes}, 0
    return {c: counts.get(c, 0) / total for c in codes}, total

def desirability_score(counts, weights, codes):
    """S(p) = sum_o w_o p(o)."""
    p, _ = normalize(counts, codes)
    return sum(weights[c] * p[c] for c in codes)


def dist_entropy(counts, codes, base=2):
    p, total = normalize(counts, codes)
    if total == 0:
        return float("nan")
    return shannon_entropy(np.array([p[c] for c in codes]), base=base)  # scipy handles zeros

In [ ]:
def baseline_gap(model_base_list, human_list):
    mb, hu = to_dict(model_base_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)

    rows = []
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:                       # non-directional: no gap, stays in JSD
            continue
        weights, norm = built

        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm

        rows.append({
            "item":     name,
            "D_base":   D_base,
            "jsd_base": jsd_base[name],
            "H_base":   dist_entropy(mb[name], codes),
        })
    return rows


def baseline_summary(rows, jsd_base_all=None):
    absD_b = [abs(r["D_base"]) for r in rows]
    out = {
        "n_items":        len(rows),
        "mean_D_base":    np.mean([r["D_base"] for r in rows]),
        "mean_absD_base": np.mean(absD_b),
        "mean_H_base":    np.nanmean([r["H_base"] for r in rows]),
        "mean_jsd_base":  np.mean([r["jsd_base"] for r in rows]),   # JSD over directional items only
    }
    if jsd_base_all is not None:   # JSD over all 10 items, matches Tables 12/14/16
        out["mean_jsd_base_all"] = np.mean(list(jsd_base_all.values()))
    return out

In [22]:
rows = baseline_gap(replicate_results, human_results)

jsd_base_all = jsd_divergence_between_sets(replicate_results, human_results)
summary = baseline_summary(rows, jsd_base_all=jsd_base_all)

print(f"{model_name}, Replicate (baseline)")
for k, v in summary.items():
    print(f"  {k:18s} {v}")

gpt4.1, Replicate (baseline)
  n_items            8
  mean_D_base        0.16265667086504473
  mean_absD_base     0.1855615196955233
  mean_H_base        0.9372146268589006
  mean_jsd_base      0.07093678640021174
  mean_jsd_base_all  0.07093678640021174


In [ ]:
conditions = {
    "Reformulated": {"data": reformulated_results},
}

def subset(count_list, names):
    """Keep only the given item names from a [{name:{...}}] list."""
    keep = set(names)
    return [d for d in count_list if list(d)[0] in keep]

REVERSE_MAPPINGS = {
    "Gay Marriage":     {1: 3, 3: 1},
    "Refugee Allowing": {1: 2, 2: 1},
    "Income Inequality":{1: 2, 2: 1},
    "Gender Role":      {1: 2, 2: 1},
    "Climate Change":   {1: 5, 2: 4, 4: 2, 5: 1},
    "Race Diversity":   {1: 2, 2: 1},
}

def apply_reverse(count_list):
    d = to_dict(count_list)
    out = {}
    for name, counts in d.items():
        m = REVERSE_MAPPINGS.get(name)
        if m is None:
            out[name] = dict(counts)
        else:
            remapped = {}
            for code, n in counts.items():
                tgt = m.get(code, code)          # map code, or keep if not in map
                remapped[tgt] = remapped.get(tgt, 0) + n
            out[name] = remapped
    return [{k: v} for k, v in out.items()]

# original function of reverse
from copy import deepcopy
def reverse_answers(rever_coded_V2_results):
    d = {list(x)[0]: deepcopy(list(x.values())[0]) for x in rever_coded_V2_results}  # copy inner dicts
    src = deepcopy(d)
    d["Gay Marriage"][1], d["Gay Marriage"][3] = src["Gay Marriage"][3], src["Gay Marriage"][1]
    d["Refugee Allowing"][1], d["Refugee Allowing"][2] = src["Refugee Allowing"][2], src["Refugee Allowing"][1]
    d["Income Inequality"][1], d["Income Inequality"][2] = src["Income Inequality"][2], src["Income Inequality"][1]
    d["Gender Role"][1], d["Gender Role"][2] = src["Gender Role"][2], src["Gender Role"][1]
    d["Climate Change"][1], d["Climate Change"][5] = src["Climate Change"][5], src["Climate Change"][1]
    d["Climate Change"][2], d["Climate Change"][4] = src["Climate Change"][4], src["Climate Change"][2]
    d["Race Diversity"][1], d["Race Diversity"][2] = src["Race Diversity"][2], src["Race Diversity"][1]
    return [{k: v} for k, v in d.items()]

def run_all(baseline, conditions, human, name_dict):
    results = {}
    for cname, cfg in conditions.items():
        cond = cfg["data"]
        if cfg.get("remap"):
            # cond = apply_reverse(cond)                       # flip back to original coding
            cond = reverse_answers(cond)                       # flip back to original coding
        if "items" in cfg:                                   # reverse-coded: 6-item subset
            names = [name_dict[q] for q in cfg["items"]]
            base = subset(baseline, names)
            cond = subset(cond, names)
        else:
            base = baseline
        rows   = gap_analysis(base, cond, human)
        jb_all = jsd_divergence_between_sets(base, human)    # baseline matched to the item set
        jc_all = jsd_divergence_between_sets(cond, human)
        results[cname] = {"summary": gap_summary(rows, jb_all, jc_all),
                          "rows": rows, "n_gap": len(rows)}
    return results

res = run_all(replicate_results, conditions, human_results, name_dict)
for cname, r in res.items():
    s = r["summary"]
    print(f"{model_name} | {cname}: dJSD {s['mean_jsd_base']:.3f}->{s['mean_jsd_cond']:.3f}, "
          f"|D| {s['mean_absD_base']:.3f}->{s['mean_absD_cond']:.3f}  (gap n={r['n_gap']})")
    
for cname, r in res.items():
    s = r["summary"]

    delta_jsd = s['mean_jsd_cond'] - s['mean_jsd_base']
    delta_absD = s['mean_absD_cond'] - s['mean_absD_base']

    print(
        f"{model_name} | {cname}: "
        f"ΔJSD={delta_jsd:+.3f}, "
        f"Δ|D|={delta_absD:+.3f} "
        f"(gap n={r['n_gap']})"
    )

gpt4.1 | Reformulated: dJSD 0.071->0.060, |D| 0.186->0.134  (gap n=8)
gpt4.1 | Reformulated: ΔJSD=-0.011, Δ|D|=-0.051 (gap n=8)


In [ ]:
res_gpt = res

In [27]:
def gap_analysis(model_base_list, model_cond_list, human_list):
    """Per-item gap and JSD, baseline vs one condition.
    Sign: condition - base, negative = improvement."""
    mb, mc, hu = to_dict(model_base_list), to_dict(model_cond_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)
    jsd_cond = jsd_divergence_between_sets(model_cond_list, human_list)
    rows = []
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:                       # non-directional: skip gap, stays in JSD
            continue
        weights, norm = built
        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm
        D_cond = (desirability_score(mc[name], weights, codes) - S_h) / norm
        rows.append({
            "item":       name,
            "D_base":     D_base, "D_cond": D_cond,
            "delta_absD": abs(D_cond) - abs(D_base),        # cond - base, NEG = gap shrank
            "delta_jsd":  jsd_cond[name] - jsd_base[name],  # cond - base, NEG = JSD improved
            "jsd_base":   jsd_base[name], "jsd_cond": jsd_cond[name],
        })
    return rows

def gap_summary(rows, jsd_base_all=None, jsd_cond_all=None):
    out = {
        "n_items":        len(rows),
        "mean_D_base":    np.mean([r["D_base"] for r in rows]),
        "mean_D_cond":    np.mean([r["D_cond"] for r in rows]),
        "mean_absD_base": np.mean([abs(r["D_base"]) for r in rows]),
        "mean_absD_cond": np.mean([abs(r["D_cond"]) for r in rows]),
        "mean_jsd_base":  np.mean([r["jsd_base"] for r in rows]),   # n = len(rows)
        "mean_jsd_cond":  np.mean([r["jsd_cond"] for r in rows]),
    }
    if jsd_base_all is not None:
        out["mean_jsd_base_all"] = np.mean(list(jsd_base_all.values()))  # n=10, matches tables
    if jsd_cond_all is not None:
        out["mean_jsd_cond_all"] = np.mean(list(jsd_cond_all.values()))
    return out

In [28]:
def print_gap_table(cname, r):
    rows, s = r["rows"], r["summary"]
    print(f"\n{cname}  (baseline -> {cname}, negative = improvement)")
    print(f"  {'item':<18} {'D_base':>8} {'D_cond':>8} {'dJSD':>8} {'d|D|':>8}")
    for x in rows:
        print(f"  {x['item']:<18} {x['D_base']:>+8.3f} {x['D_cond']:>+8.3f} "
              f"{x['delta_jsd']:>+8.3f} {x['delta_absD']:>+8.3f}")
    print(f"  {'MEAN':<18} {s['mean_D_base']:>+8.3f} {s['mean_D_cond']:>+8.3f} "
          f"{s['mean_jsd_cond']-s['mean_jsd_base']:>+8.3f} "
          f"{s['mean_absD_cond']-s['mean_absD_base']:>+8.3f}")
    print(f"  n={s['n_items']}")

for cname, r in res.items():
    print(model_name,'ANES 2024')
    print_gap_table(cname, r)
    print()

gpt4.1 ANES 2024

Reformulated  (baseline -> Reformulated, negative = improvement)
  item                 D_base   D_cond     dJSD     d|D|
  Gay Marriage         +0.156   +0.074   -0.016   -0.082
  Income Inequality    +0.173   +0.162   +0.074   -0.010
  Gender Role          -0.057   +0.088   -0.007   +0.030
  Climate Change       +0.145   +0.003   -0.051   -0.142
  Gun Regulation       -0.034   -0.090   +0.022   +0.056
  Drug Addiction       +0.242   +0.150   -0.010   -0.092
  Race Diversity       +0.507   +0.320   -0.102   -0.186
  Health Insurance     +0.170   +0.187   +0.005   +0.017
  MEAN                 +0.163   +0.112   -0.011   -0.051
  n=8



## Bootstrap intervals and significance tests (ANES 2024, Reformulated)

In [ ]:
# Bootstrap machinery
import numpy as np
import pandas as pd
import zlib

N_BOOT = 10_000
SEED = 20250701          # same seed as the ANES 2020 notebook


def _cvec(counts, codes):
    return np.array([counts.get(c, 0) for c in codes], dtype=float)


def _resample(cvec, rng, n_rep):
    """(n_rep, k) matrix of bootstrap-resampled counts."""
    n = cvec.sum()
    if n == 0:
        return np.zeros((n_rep, len(cvec)))
    return rng.multinomial(int(n), cvec / n, size=n_rep).astype(float)


def _item_rng(name, seed=SEED):
    """Child RNG per item. Both drivers below call this with the same name, so
    an item's human/baseline/condition draws are identical in the gap and the
    divergence bootstrap: the two metrics are computed on the same replicates."""
    return np.random.default_rng([seed, zlib.crc32(name.encode())])


def _S(cmat, w):
    """Desirability-weighted score per replicate: S = sum_o w_o p_o."""
    tot = cmat.sum(axis=1, keepdims=True)
    tot = np.where(tot == 0, np.nan, tot)
    return (cmat / tot) @ w


def _jsd_rows(Pm, Qm, base=2.0):
    """Row-wise JSD between two (n_rep, k) count matrices, matching
    jsd_divergence_between_sets(): clip at 1e-10, then renormalise as
    scipy.stats.entropy does internally."""
    p = Pm / Pm.sum(axis=1, keepdims=True)
    q = Qm / Qm.sum(axis=1, keepdims=True)
    p = np.clip(p, 1e-10, 1.0); q = np.clip(q, 1e-10, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    q = q / q.sum(axis=1, keepdims=True)
    m = 0.5 * (p + q); m = m / m.sum(axis=1, keepdims=True)
    kl = lambda a: np.sum(a * np.log(a / m), axis=1)
    return 0.5 * (kl(p) + kl(q)) / np.log(base)


def _jsd_point(counts_a, counts_b, codes):
    """Point estimate on observed counts (silicon first, as in run_all)."""
    return float(_jsd_rows(_cvec(counts_a, codes)[None, :], _cvec(counts_b, codes)[None, :])[0])


def boot_p_two_sided(delta):
    """Floor is 2/(n+1), i.e. 0.0002 at n = 10,000."""
    n = len(delta)
    k = min(int(np.sum(delta <= 0)), int(np.sum(delta >= 0)))
    return min(1.0, 2.0 * (1 + k) / (n + 1))


def boot_p_one_sided(x, tail):
    """tail='less'    -> H0: x >= 0  (improvement, delta < 0)
       tail='greater' -> H0: x <= 0  (baseline SDB, D > 0)"""
    n = len(x)
    k = int(np.sum(x >= 0)) if tail == "less" else int(np.sum(x <= 0))
    return (1 + k) / (n + 1)


def bh_adjust(p):
    """Benjamini-Hochberg step-up, returned in input order."""
    p = np.asarray(p, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(adj, 0, 1)
    return out

In [ ]:
# Paired bootstrap for both metrics
def bootstrap_all(baseline, conditions, human, name_dict, n_rep=N_BOOT, seed=SEED):
    hu, mb = to_dict(human), to_dict(baseline)

    prepared = {}
    for cname, cfg in conditions.items():
        cond = cfg["data"]
        if cfg.get("remap"):
            cond = reverse_answers(cond)
        keep = set(name_dict[q] for q in cfg["items"]) if "items" in cfg else None
        prepared[cname] = (to_dict(cond), keep)

    base_rows, cond_rows = [], []
    acc = {}                                  # condition -> list of per-item replicate arrays

    for name in mb:
        codes = option_codes(name_to_qid[name])
        rng = _item_rng(name, seed)

        H = _resample(_cvec(hu[name], codes), rng, n_rep)     # shared
        B = _resample(_cvec(mb[name], codes), rng, n_rep)     # shared

        J_b = _jsd_rows(B, H)
        J_b_pt = _jsd_point(mb[name], hu[name], codes)

        built = build_weights(desirability_spec[name], codes)
        directional = built is not None
        if directional:
            weights, norm = built
            w = np.array([weights[c] for c in codes], dtype=float)
            S_h = _S(H, w)
            D_b = (_S(B, w) - S_h) / norm
            D_b_pt = (desirability_score(mb[name], weights, codes)
                      - desirability_score(hu[name], weights, codes)) / norm
        else:
            D_b = D_b_pt = None

        n_sil = int(_cvec(mb[name], codes).sum())
        row = {"item": name, "jsd_base": J_b_pt,
               "jsd_ci_lo": float(np.percentile(J_b, 2.5)),
               "jsd_ci_hi": float(np.percentile(J_b, 97.5)),
               "n_silicon": n_sil,
               "collapsed": bool(n_sil > 0 and float(_cvec(mb[name], codes).max()) == n_sil)}
        if directional:
            row.update({"D_base": D_b_pt,
                        "D_ci_lo": float(np.nanpercentile(D_b, 2.5)),
                        "D_ci_hi": float(np.nanpercentile(D_b, 97.5)),
                        "p_D_gt_0": boot_p_one_sided(D_b, tail="greater"),
                        "fold_risk": bool(np.nanpercentile(D_b, 2.5) < 0 < np.nanpercentile(D_b, 97.5))})
        base_rows.append(row)

        for cname, (cd, keep) in prepared.items():
            if name not in cd or (keep is not None and name not in keep):
                continue
            C = _resample(_cvec(cd[name], codes), rng, n_rep)
            dJ = _jsd_rows(C, H) - J_b
            J_c_pt = _jsd_point(cd[name], hu[name], codes)

            r = {"item": name, "condition": cname,
                 "jsd_base": J_b_pt, "jsd_cond": J_c_pt,
                 "delta_jsd": J_c_pt - J_b_pt,
                 "jsd_ci_lo": float(np.percentile(dJ, 2.5)),
                 "jsd_ci_hi": float(np.percentile(dJ, 97.5)),
                 "p_jsd": boot_p_two_sided(dJ),
                 "p_jsd_1s": boot_p_one_sided(dJ, tail="less")}

            if directional:
                D_c = (_S(C, w) - S_h) / norm
                dA = np.abs(D_c) - np.abs(D_b)
                D_c_pt = (desirability_score(cd[name], weights, codes)
                          - desirability_score(hu[name], weights, codes)) / norm
                cv = _cvec(cd[name], codes)
                r.update({"D_base": D_b_pt, "D_cond": D_c_pt,
                          "delta_absD": abs(D_c_pt) - abs(D_b_pt),
                          "D_ci_lo": float(np.nanpercentile(dA, 2.5)),
                          "D_ci_hi": float(np.nanpercentile(dA, 97.5)),
                          "p_absD": boot_p_two_sided(dA),
                          "p_absD_1s": boot_p_one_sided(dA, tail="less"),
                          "sign_flip": float(np.mean(np.sign(D_c) != np.sign(D_b))),
                          "collapsed_cond": bool(cv.sum() > 0 and float(cv.max()) == cv.sum())})
                acc.setdefault(cname, {"dJ": [], "dA": []})["dA"].append(dA)
            acc.setdefault(cname, {"dJ": [], "dA": []})["dJ"].append(dJ)
            cond_rows.append(r)

    base_df = pd.DataFrame(base_rows)
    cond_df = pd.DataFrame(cond_rows)

    # Separate BH families for the two metrics, each over all item-condition cells.
    cond_df["p_jsd_BH"] = bh_adjust(cond_df["p_jsd"].values)
    cond_df["sig_jsd"] = cond_df["p_jsd_BH"] <= 0.05
    m = cond_df["p_absD"].notna().values
    adj = np.full(len(cond_df), np.nan)
    adj[m] = bh_adjust(cond_df.loc[m, "p_absD"].values)
    cond_df["p_absD_BH"] = adj
    cond_df["sig_absD"] = cond_df["p_absD_BH"] <= 0.05
    if "p_D_gt_0" in base_df:
        mb_ = base_df["p_D_gt_0"].notna().values
        a2 = np.full(len(base_df), np.nan)
        a2[mb_] = bh_adjust(base_df.loc[mb_, "p_D_gt_0"].values)
        base_df["p_D_BH"] = a2

    # Joint criterion: both metrics improve. Replicates are paired, so the
    # intersection-union p-value is the max of the two one-sided p-values and
    # needs no further correction across the two components.
    cond_df["p_joint"] = cond_df[["p_jsd_1s", "p_absD_1s"]].max(axis=1)
    mj = cond_df["p_joint"].notna().values
    a3 = np.full(len(cond_df), np.nan)
    a3[mj] = bh_adjust(cond_df.loc[mj, "p_joint"].values)
    cond_df["p_joint_BH"] = a3
    cond_df["joint"] = ((cond_df["delta_jsd"] < 0) & (cond_df["delta_absD"] < 0)
                        & (cond_df["p_joint_BH"] <= 0.05))

    mean_rows = []
    for cname, d in acc.items():
        mJ = np.mean(np.vstack(d["dJ"]), axis=0)
        row = {"condition": cname, "n_items_jsd": len(d["dJ"]),
               "mean_delta_jsd": float(np.mean(mJ)),
               "jsd_ci_lo": float(np.percentile(mJ, 2.5)),
               "jsd_ci_hi": float(np.percentile(mJ, 97.5)),
               "p_jsd": boot_p_two_sided(mJ)}
        if d["dA"]:
            mA = np.nanmean(np.vstack(d["dA"]), axis=0)
            row.update({"n_items_absD": len(d["dA"]),
                        "mean_delta_absD": float(np.nanmean(mA)),
                        "D_ci_lo": float(np.nanpercentile(mA, 2.5)),
                        "D_ci_hi": float(np.nanpercentile(mA, 97.5)),
                        "p_absD": boot_p_two_sided(mA)})
        mean_rows.append(row)
    return base_df, cond_df, pd.DataFrame(mean_rows)


base_df, cond_df, mean_df = bootstrap_all(replicate_results, conditions, human_results, name_dict)

# verification: point estimates must reproduce the existing functions
_ref = jsd_divergence_between_sets(replicate_results, human_results)
_d1 = max(abs(_ref[r["item"]] - r["jsd_base"]) for _, r in base_df.iterrows())
_gap = {r["item"]: r for r in baseline_gap(replicate_results, human_results)}
_d2 = max(abs(_gap[r["item"]]["D_base"] - r["D_base"]) for _, r in base_df.iterrows()
          if pd.notna(r.get("D_base")))
print(f"check vs jsd_divergence_between_sets: max diff = {_d1:.2e}  {'OK' if _d1 < 1e-9 else 'MISMATCH'}")
print(f"check vs baseline_gap:                max diff = {_d2:.2e}  {'OK' if _d2 < 1e-9 else 'MISMATCH'}")

print(f"\n=== {model_name} (ANES 2024): baseline JSD and signed gap D ===")
print(base_df[["item", "jsd_base", "jsd_ci_lo", "jsd_ci_hi", "D_base", "D_ci_lo", "D_ci_hi",
               "p_D_gt_0", "p_D_BH", "fold_risk", "collapsed", "n_silicon"]].to_string(index=False))

for c in cond_df["condition"].unique():
    s = cond_df[cond_df["condition"] == c]
    print(f"\n=== {model_name} (ANES 2024): {c} vs Replicate ===")
    print(s[["item", "delta_jsd", "jsd_ci_lo", "jsd_ci_hi", "p_jsd_BH", "sig_jsd",
             "D_base", "D_cond", "delta_absD", "D_ci_lo", "D_ci_hi", "p_absD_BH", "sig_absD",
             "p_joint_BH", "joint", "fold_risk" if "fold_risk" in s else "sign_flip",
             "collapsed_cond"]].to_string(index=False))

print(f"\n=== {model_name} (ANES 2024): condition means ===")
print(mean_df.to_string(index=False))

check vs jsd_divergence_between_sets: max diff = 2.22e-16  OK
check vs baseline_gap:                max diff = 0.00e+00  OK

=== gpt4.1 (ANES 2024): baseline JSD and signed gap D ===
             item  jsd_base  jsd_ci_lo  jsd_ci_hi    D_base   D_ci_lo   D_ci_hi  p_D_gt_0   p_D_BH  fold_risk  collapsed  n_silicon
     Gay Marriage  0.021406   0.016069   0.027526  0.156238  0.134106  0.177527    0.0001 0.000133      False      False       5521
Income Inequality  0.022616   0.018090   0.027911  0.172786  0.153768  0.192049    0.0001 0.000133      False      False       5521
      Gender Role  0.025595   0.021104   0.030687 -0.057413 -0.064650 -0.050314    1.0000 1.000000      False      False       5521
   Climate Change  0.081069   0.073877   0.088970  0.145010  0.133495  0.156467    0.0001 0.000133      False      False       5521
   Gun Regulation  0.000891   0.000222   0.002265 -0.034206 -0.053735 -0.014917    0.9997 1.000000      False      False       5521
   Drug Addiction  0.1228